In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from Bio import SeqIO
from tqdm import tqdm
import json
import os
import logging
import numpy as np
import matplotlib.patches as patches
import random
from scipy.stats import mannwhitneyu
from sklearn.preprocessing import MinMaxScaler

# Import all our custom pipeline modules
from instanexus import preprocessing
from instanexus import assembly
from instanexus import visualization
from instanexus import helpers


# Set up logging to see the pipeline's progress
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
os.chdir('../../../../')

print(f"Current working directory: {os.getcwd()}")

In [ ]:
FIGURES_DIR = Path("figures")
print(FIGURES_DIR)

In [ ]:
# Path to the new raw data you want to test

INPUT_CSV = "inputs/ma1_all.csv"
METADATA_PATH = "json/sample_metadata.json"
CONTAMINANTS_PATH = "fasta/contaminants.fasta"
RUN_NAME = Path(INPUT_CSV).stem

# Assembly params
ASSEMBLY_MODE = "dbg_weighted"
CHAIN = "heavy"
REFERENCE_MODE = True
KMER_SIZE = 6
MIN_OVERLAP = 3
SIZE_THRESHOLD = 0.2
CONFIDENCE_THRESHOLD = 0.9
MIN_LENGTH = 7
MAX_LENGTH = 20
FDR_THRESHOLD = 0.5
MIN_IDENTITY = 0.8
MAX_MISMATCHES = 0

# Clustering params
MIN_SEQ_ID = 0.85
COVERAGE = 0.8

In [ ]:
RUN_NAME

In [ ]:
#base_output_folder = Path(BASE_OUTPUT_FOLDER) / RUN_NAME

# Build the unique experiment folder name
folder_name_parts = [f"{ASSEMBLY_MODE}"]

if CONFIDENCE_THRESHOLD is not None:
    folder_name_parts.append(f"c{CONFIDENCE_THRESHOLD}")

if "dbg" in ASSEMBLY_MODE:
    folder_name_parts.append(f"ks{KMER_SIZE}")

folder_name_parts.append(f"mo{MIN_OVERLAP}")
folder_name_parts.append(f"ts{SIZE_THRESHOLD}")

if REFERENCE_MODE:
    folder_name_parts.extend([f"mi{MIN_IDENTITY}", f"mm{MAX_MISMATCHES}"])

run_folder_name = "_".join(folder_name_parts)
#experiment_folder = base_output_folder / run_folder_name

run_id_str = f"[{RUN_NAME} @ {run_folder_name}]"

logger.info(f"Pipeline starting for run: {run_id_str}")

In [ ]:
sample_metadata = preprocessing.get_sample_metadata(
    run=RUN_NAME, 
    chain=CHAIN, 
    json_path=METADATA_PATH
)

In [ ]:
proteases = sample_metadata["proteases"]
protein = sample_metadata["protein"]
protein_norm = preprocessing.normalize_sequence(protein)

In [ ]:
print(f"Sample uses proteases: {proteases}")
print(f"Protein sequence length: {len(protein)} amino acids")
print(f"Normalized protein sequence: {protein_norm}")

In [ ]:
original_data = pd.read_csv(INPUT_CSV)

In [ ]:
#extend the limit so I can see all the info of original_data['experiment_name']
pd.set_option('display.max_colwidth', None)

In [ ]:
original_data.columns

In [ ]:
original_data["protease"] = original_data["experiment_name"].apply(
    lambda name: preprocessing.extract_protease(name, proteases)
)


In [ ]:
original_data = original_data.dropna(subset=["preds"])


In [ ]:
original_data["cleaned_preds"] = original_data["preds"].apply(preprocessing.remove_modifications)

In [ ]:
original_data["conf"] = original_data["log_probs"].apply(np.exp)

In [ ]:
original_data["number_amino_acids"] = original_data["cleaned_preds"].apply(len)

In [ ]:
print(MIN_LENGTH)

In [ ]:
MAX_LENGTH

In [ ]:
# remove all the raw that in the column "protease" contain "Vesuvious" and "Krakatoa"
original_data = original_data[~original_data["protease"].str.contains("Vesuvious|Krakatoa")]

In [ ]:
data_filtered = original_data[(original_data["conf"] > CONFIDENCE_THRESHOLD) & (original_data["number_amino_acids"] >= MIN_LENGTH) & (original_data["number_amino_acids"] <= MAX_LENGTH)]


In [ ]:
sequences = data_filtered['cleaned_preds'].tolist()

In [ ]:
assembler = assembly.Assembler(
    mode="dbg_weighted",
    kmer_size=7,
    min_overlap=3,
    size_threshold=5,
    min_weight=2,
    refine_rounds=5
)

In [ ]:
scaffolds = assembler.run(sequences=sequences, df_full=data_filtered)

In [ ]:
mapped_scaffolds = visualization.process_protein_contigs_scaffold(
    scaffolds, protein_norm, 10, 0.7)

In [ ]:
print(RUN_NAME, CHAIN)

In [ ]:
mapped_scaffolds

In [ ]:
def mapping_sequences(
    mapped_sequences,
    prot_seq,
    category,
    run_name=None,      
    chain_type=None,    
    cdr_data=None,      
    config_json_path="json/colors.json",
    output_folder=".",
    output_file=None,
    show_figure=False,
):
    visualization.set_publication_style()
    
    if cdr_data and run_name and chain_type:
        print(f"DEBUG: Looking for CDRs -> Run: {run_name}, Chain: {chain_type}")
    else:
        print("DEBUG: No CDR info provided (Standard mode)")

    try:
        with open(config_json_path, 'r') as f:
            color_data = json.load(f)
        main_color = color_data.get(category, {}).get("scaffold", "#1f78b4")
    except Exception:
        main_color = "#1f78b4"

    fig_width, fig_height = visualization.get_figsize(width_ratio=3)
    common_height = 0.3
    track_spacing = 0.45
    base_y_offset = 0.6

    _, ax = plt.subplots(figsize=(fig_width, fig_height))

    ax.add_patch(patches.Rectangle(
        (0, 0), len(prot_seq), common_height,
        linewidth=0, facecolor='#e6f0ef', zorder=0
    ))

    cdr_colors = {"cdr1": "#FFB347", "cdr2": "#77DD77", "cdr3": "#89CFF0"}
    active_cdrs = {}

    if cdr_data and run_name and chain_type:
        if run_name in cdr_data:
            for entry in cdr_data[run_name]:
                if entry.get("chain", "").lower() == chain_type.lower():
                    active_cdrs = entry.get("cdrs", {})
                    print(f"DEBUG: Found CDRs: {list(active_cdrs.keys())}") # Debug
                    break
    
    tracks = {}
    colors = {
        "match": main_color,
        "mismatch": "#b30000",
        "D_to_N": "#000000",
        "E_to_Q": "#A8A29E",
    }

    for seq, mapping in tqdm(mapped_sequences, desc=f"Mapping {category}"):
        start_index, end_index, mismatches, _ = mapping
        placed = False
        for track_num in sorted(tracks.keys()):
            if not any(max(s, start_index) < min(e, end_index) for s, e in tracks[track_num]):
                tracks[track_num].append((start_index, end_index))
                current_track_num = track_num
                placed = True
                break
        if not placed:
            current_track_num = len(tracks)
            tracks[current_track_num] = [(start_index, end_index)]

        current_y = base_y_offset + (current_track_num * track_spacing)
        
        ax.add_patch(patches.Rectangle(
            (start_index, current_y), end_index - start_index, common_height,
            linewidth=0.8, edgecolor='white', facecolor=colors["match"], alpha=0.9, zorder=10
        ))

        for mismatch in mismatches:
            abs_index = start_index + mismatch
            if abs_index >= len(prot_seq) or mismatch >= len(seq): continue
            ref_aa = prot_seq[abs_index]
            query_aa = seq[mismatch]
            if query_aa == "D" and ref_aa == "N": mut_color = colors["D_to_N"]
            elif query_aa == "E" and ref_aa == "Q": mut_color = colors["E_to_Q"]
            else: mut_color = colors["mismatch"]
            
            ax.add_patch(patches.Rectangle(
                (abs_index, current_y), 1, common_height,
                linewidth=0, facecolor=mut_color, zorder=15
            ))

    max_track = len(tracks) if tracks else 0
    max_y = base_y_offset + (max_track * track_spacing) + 0.5
    
    for cdr_name, details in active_cdrs.items():
        if not details or "start" not in details: continue
        
        start = details["start"] - 1
        end = details["end"]
        width = end - start
        
        c_color = cdr_colors.get(cdr_name.lower(), "gray")
        
        cdr_rect = patches.Rectangle(
            (start, -0.5), width, max_y + 1, 
            facecolor=c_color, alpha=0.3, zorder=1, linewidth=0
        )
        ax.add_patch(cdr_rect)
        
        mid_point = (start + end) / 2
        ax.text(mid_point, max_y + 0.1, cdr_name.upper(), 
                ha='center', va='bottom', fontsize=9, fontweight='normal', 
                color='black', zorder=20)

    ax.set_xlim(0, len(prot_seq))
    y_upper_limit = max_y + 0.6 if active_cdrs else max_y
    ax.set_ylim(-0.1, y_upper_limit)
    
    ax.set_xlabel("Residue position")
    ax.set_yticks([])
    sns.despine(left=True)

    legend_patches = [
        patches.Patch(color=colors["match"], label=f"Match"),
        patches.Patch(color=colors["mismatch"], label="Mismatch"),
        patches.Patch(color=colors["D_to_N"], label="D \u2192 N"),
        patches.Patch(color=colors["E_to_Q"], label="E \u2192 Q"),
    ]
    ax.legend(handles=legend_patches, loc='upper center', bbox_to_anchor=(0.5, 1.25), ncol=4, frameon=False)

    plt.tight_layout()

    if output_file:
        try:
            os.makedirs(output_folder, exist_ok=True)
            save_path = os.path.join(output_folder, output_file)
            plt.savefig(save_path, format='svg', bbox_inches='tight')
            print(f"Saved: {save_path}")
        except Exception:
            plt.savefig(output_file, format='svg', bbox_inches='tight')

    if show_figure:
        plt.show()
    plt.close()

In [ ]:
with open("json/cdrs_antibodies.json", "r") as f:
    cdr_dict = json.load(f)

In [ ]:
print(CONFIDENCE_THRESHOLD)

In [ ]:
mapping_sequences(
    mapped_scaffolds,
    protein_norm,
    category="antibodies",
    run_name=RUN_NAME,
    chain_type=CHAIN,
    cdr_data=cdr_dict,
    config_json_path="json/colors.json",
    output_folder=FIGURES_DIR,
    output_file=f"fig5c_{RUN_NAME}_{CHAIN}_{ASSEMBLY_MODE}_{CONFIDENCE_THRESHOLD}_scaffold_mapping_2_prot_removed.svg",
    show_figure=True
)